In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [2]:
! pip install langchain-huggingface

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
)

chat = ChatHuggingFace(llm=llm)

c:\Users\somil\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
llm = chat

In [5]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [6]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [7]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [8]:
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 30, 'total_tokens': 40}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f749f-c899-7013-9213-ae51493eaeee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 10, 'total_tokens': 40})

In [9]:
llm_with_tools = llm.bind_tools([multiply])

In [10]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with any questions or tasks you have. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 215, 'total_tokens': 256}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f74a0-3de3-7b40-b230-ef55b2306abb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 215, 'output_tokens': 41, 'total_tokens': 256})

In [11]:
query = HumanMessage('can you multiply 3 with 1000')

In [12]:
messages = [query]

In [13]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [14]:
result = llm_with_tools.invoke(messages)

In [15]:
messages.append(result)

In [16]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply', 'description': None}, 'id': 'call_5hl3nsm86lx77h6imnfheze9', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 222, 'total_tokens': 250}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f74a0-fabc-7c51-9a4e-c3426da2f17f-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_5hl3nsm86lx77h6imnfheze9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 28, 'total_tokens': 250})]

In [17]:
tool_result = multiply.invoke(result.tool_calls[0])

In [18]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='call_5hl3nsm86lx77h6imnfheze9')

In [20]:
messages.append(tool_result)

In [21]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply', 'description': None}, 'id': 'call_5hl3nsm86lx77h6imnfheze9', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 222, 'total_tokens': 250}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f74a0-fabc-7c51-9a4e-c3426da2f17f-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_5hl3nsm86lx77h6imnfheze9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 28, 'total_tokens': 250}),
 ToolMessage(content='3000', name='multiply', tool_call_id='call_5hl3nsm86lx77h6imnfheze9')]

In [19]:
llm_with_tools.invoke(messages).content

'The product of 3 and 1000 is 3000.'

## Currency Conversion Tool

In [43]:
# tool create

from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def conversion_factor(base_currency : str, target_currency : str)->float:
    """ This function fetches current conversion factor between base_curreny and target_currency"""
    url =f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
    response = requests.get(url)

    return response.json()



In [44]:
result = conversion_factor.invoke(
    {
        "base_currency": "USD",
        "target_currency": "INR"
    }
)

print(result)

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1784332802, 'time_last_update_utc': 'Sat, 18 Jul 2026 00:00:02 +0000', 'time_next_update_unix': 1784419202, 'time_next_update_utc': 'Sun, 19 Jul 2026 00:00:02 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 96.4102}


In [45]:
@tool
def convert(base_currency_value : int, conversion_rate: Annotated[float,InjectedToolArg]) -> float:
    """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """
    
    return base_currency_value * conversion_rate
    

In [46]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [47]:
conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784332802,
 'time_last_update_utc': 'Sat, 18 Jul 2026 00:00:02 +0000',
 'time_next_update_unix': 1784419202,
 'time_next_update_utc': 'Sun, 19 Jul 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.4102}

In [48]:
convert.invoke({'base_currency_value' : 1200, 'conversion_rate':96.4102})

115692.24

In [49]:
llm_with_tools = llm.bind_tools([conversion_factor, convert])

In [59]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd')]

In [60]:
ai_message = llm_with_tools.invoke(messages)

In [61]:
messages.append(ai_message)

In [62]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'conversion_factor', 'description': None}, 'id': 'call_5ee9f8hyu9l4zwuuc799sapw', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'call_z62ow4kjdg1h1gj3x0vn5rpu', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 342, 'total_tokens': 393}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f74e5-b6bf-7762-a3aa-ca032cdfbf69-0', tool_calls=[{'name': 'conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_5ee9f8hyu9l4zwuuc799sapw', 'type':

In [63]:
ai_message.tool_calls

[{'name': 'conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_5ee9f8hyu9l4zwuuc799sapw',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_z62ow4kjdg1h1gj3x0vn5rpu',
  'type': 'tool_call'}]

In [64]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'conversion_factor':
        tool_message1 = conversion_factor.invoke(tool_call)
        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)


In [65]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'conversion_factor', 'description': None}, 'id': 'call_5ee9f8hyu9l4zwuuc799sapw', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'call_z62ow4kjdg1h1gj3x0vn5rpu', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 342, 'total_tokens': 393}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f74e5-b6bf-7762-a3aa-ca032cdfbf69-0', tool_calls=[{'name': 'conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_5ee9f8hyu9l4zwuuc799sapw', 'type':

In [66]:
llm_with_tools.invoke(messages).content

'The current conversion rate from USD to INR is approximately 96.4102. Therefore, 10 INR is equivalent to about 0.1037 USD (which is calculated as 10 INR divided by 96.4102).'